# 09 — 4개 앙상블: Weighted XGBoost E + LR E + Hotspot×3/Position-bin LR + Exact-position LightGBM

- 담당 ③ 안주형. **08번 원본 변경 없음**. 팀 공통 `get_folds` / 고정 홀드아웃 유지.
- 4개 모델 확률 **동일 비중 25%**. 3번째 모델(Weighted XGB E + E/Position-bin LR, 50:50)과 동일 개발 Fold/홀드아웃에서 비교.
- 핫스팟 LR은 시윤님 03번의 **Hotspot×3 + Position-bin(50)** 구성. 일반 35개 + NPM1 frameshift 1개.
- LightGBM: E + 유전자별 정확한 아미노산 위치(예: `BRAF_V600`), **Fold 학습 환자 5명 이상**만.
- 사전 탐색의 +0.015는 미검증 참고치이며 본 노트북에서 다시 계산함.
- **팀 규칙: 제출 CSV는 Git에 포함하지 않음.** 이 노트북은 CSV 생성 셀을 두지 않음.
- 08번과 동일하게 프로젝트 루트에서 실행. 데이터와 프로젝트 `src/`, `scripts/fixed_holdout_manifest.py` 필요.


In [18]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p/'src'/'config.py').is_file() and (p/'scripts'/'fixed_holdout_manifest.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('src/config.py가 있는 프로젝트 폴더에서 실행하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 노트북의 현재 작업 폴더와 무관하게 프로젝트 스크립트 로딩
import importlib.util
manifest_path = ROOT / 'scripts' / 'fixed_holdout_manifest.py'
spec = importlib.util.spec_from_file_location('fixed_holdout_manifest', manifest_path)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)
assert holdout_module.ROOT.resolve() == ROOT.resolve(), '스크립트의 ROOT 경로를 확인하세요.'
from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.validation import get_folds, run_cv
from src.evaluation import macro_f1, save_predictions, log_result
from src.features import mutation_count, mutation_type_counts, amino_acid_pair_counts, gene_truncation_flags

set_seed()
RUN_FULL_CV = True    # 팀장 승인 후 True: 전체 OOF/Test 생성 (기존 정상 파일 재사용)
SAVE_ARTIFACTS = True # True: XGB 단독/LR/앙상블 예측 파일 저장
SAVE_SUBMISSION = True # True: 최종 제출 CSV 저장
print('ROOT', ROOT, '| XGBoost', __import__('xgboost').__version__)
print('홀드아웃 스크립트:', manifest_path)

ROOT C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype | XGBoost 3.4.1
홀드아웃 스크립트: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\scripts\fixed_holdout_manifest.py


In [2]:
def select_xgb_device():
    try:
        probe = XGBClassifier(n_estimators=1, max_depth=1, tree_method='hist', device='cuda',
                              objective='binary:logistic', eval_metric='logloss', verbosity=0)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            probe.fit(np.array([[0.],[1.],[2.],[3.]],dtype=np.float32), np.array([0,0,1,1]))
        actual = json.loads(probe.get_booster().save_config())['learner']['generic_param']['device']
        if str(actual).startswith('cuda'):
            return 'cuda'
    except Exception as exc:
        print('CUDA 확인 실패:', repr(exc))
    return 'cpu'

DEVICE = select_xgb_device()
print('XGBoost device:', DEVICE)

XGBoost device: cuda


In [3]:
train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
X_test_raw = test[X_raw.columns].copy()
assert X_raw.columns.equals(X_test_raw.columns)
assert X_raw.index.equals(pd.RangeIndex(len(X_raw)))
assert len(le.classes_) == 26
# 공통 홀드아웃 먼저 읽기: CV는 개발 데이터에서만 실행합니다.
dev_idx, hold_idx = holdout_module.verify()
assert len(dev_idx) + len(hold_idx) == len(y)
assert not np.intersect1d(dev_idx, hold_idx).size
X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]
folds = get_folds(y_dev, X=X_dev_raw, classes=le.classes_)  # 팀 함수 변경 금지
print('train', X_raw.shape, 'test', X_test_raw.shape, 'classes', len(le.classes_))

def make_E(X):
    # 시윤님 02_features E와 동일한 피처 순서/변환
    parts = [binarize_mutations(X), np.log1p(mutation_count(X)),
             np.log1p(mutation_type_counts(X)), np.log1p(amino_acid_pair_counts(X)),
             gene_truncation_flags(X)]
    out = pd.concat(parts, axis=1).astype(np.float32)
    assert not out.columns.has_duplicates and out.index.equals(X.index)
    assert np.isfinite(out.to_numpy()).all()
    return out

X_E = make_E(X_raw)
X_test_E = make_E(X_test_raw)
assert X_E.columns.equals(X_test_E.columns)
assert X_E.shape[1] == 9174, f'E 피처 개수 불일치: {X_E.shape[1]}'
print('E', X_E.shape, X_test_E.shape)
X_dev_E = X_E.iloc[dev_idx].reset_index(drop=True)


VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
train (6201, 4384) test (2546, 4384) classes 26
E (6201, 9174) (2546, 9174)


## 1. 기존 08번 Position-bin LR (비교 기준)

In [4]:
# 시윤님 03_hotspot_bins.ipynb의 E+Position-bin 함수 재현 (핫스팟 미사용)
import re
from collections import Counter
from scipy.sparse import hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def make_features_E_sparse(X):
    return csr_matrix(make_E(X).to_numpy(dtype=np.float32))

def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None
# Position-bin 후보 생성 함수

from collections import defaultdict

def extract_position_bins(X, bin_size=50):
    rows = []
    cols = []
    gene_bins = set()

    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()

        for gene, value in row.items():
            if gene == "ID":
                continue

            if pd.isna(value) or str(value) == "WT":
                continue

            for variant in str(value).split():
                pos = extract_position(variant)

                if pos is not None:
                    feature = f"{gene}_bin_{pos // bin_size}"
                    patient_bins.add(feature)

        for feature in patient_bins:
            rows.append(row_idx)
            cols.append(feature)
            gene_bins.add(feature)

    return rows, cols, gene_bins


# Fold별 Position-bin 변환기 구현

from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
from collections import Counter

class PositionBinTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        _, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        counts = Counter(cols)

        self.selected_bins_ = sorted(
            feature
            for feature, count in counts.items()
            if count >= self.min_patients
        )

        self.bin_to_idx_ = {
            feature: idx
            for idx, feature in enumerate(self.selected_bins_)
        }

        return self

    def transform(self, X):
        rows, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        selected_rows = []
        selected_cols = []

        for row_idx, feature in zip(rows, cols):
            col_idx = self.bin_to_idx_.get(feature)

            if col_idx is not None:
                selected_rows.append(row_idx)
                selected_cols.append(col_idx)

        data = np.ones(len(selected_rows), dtype=np.float32)

        return csr_matrix(
            (data, (selected_rows, selected_cols)),
            shape=(len(X), len(self.selected_bins_)),
            dtype=np.float32
        )# 피처 구성 E + Position-bin CV

from scipy.sparse import hstack

class FeaturesEWithPositionBins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size,
            min_patients=self.min_patients
        )
        self.position_transformer_.fit(X)
        return self

    def transform(self, X):
        # 기존 피처 구성 E
        X_base = make_features_E_sparse(X)

        # Fold 학습 데이터에서 선택한 Position-bin
        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_position],
            format="csr",
            dtype=np.float32
        )


pipeline_E_position_LR = Pipeline([
    ("features", FeaturesEWithPositionBins(
        bin_size=50,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])



## 2. 시윤님 03번 Hotspot×3 + Position-bin LR (36개 고정 핫스팟)

In [5]:
# 유전자별 Hotspot 아미노산 위치
HOTSPOTS = {
    "IDH1": ["R132"],
    "IDH2": ["R140", "R172"],
    "BRAF": ["V600"],
    "HRAS": ["Q61", "G13", "G12"],
    "KIT": ["D816", "N822", "K642"],
    "CTNNB1": ["S33", "S37", "S45", "T41", "G34", "D32"],
    "U2AF1": ["S34"],
    "PIK3CA": ["E545", "H1047", "E542"],
    "EGFR": ["L858", "G598", "A289"],
    "RET": ["M918"],
    "RAC1": ["P29"],
    "SPOP": ["F133", "W131", "F102", "Y87"],
    "EZH2": ["Y641", "Y646"],
    "TP53": ["R175", "R248", "R273"],
    "PTEN": ["R130"],
}


assert sum(map(len,HOTSPOTS.values())) == 35

In [6]:
def make_hotspot_features(X):
    hotspot_data = {}

    # 일반 Hotspot
    for gene, positions in HOTSPOTS.items():
        if gene not in X.columns:
            raise ValueError(f"유전자 컬럼 없음: {gene}")

        tokens = X[gene].fillna("WT").astype(str).str.split()

        for position in positions:
            col_name = f"hotspot_{gene}_{position}"

            hotspot_data[col_name] = tokens.map(
                lambda variants: int(
                    any(v.startswith(position) for v in variants)
                )
            ).astype(np.int8)

    # NPM1 frameshift
    if "NPM1" not in X.columns:
        raise ValueError("유전자 컬럼 없음: NPM1")

    hotspot_data["hotspot_NPM1_fs"] = (
        X["NPM1"]
        .fillna("WT")
        .astype(str)
        .str.contains("fs", case=False, regex=False)
        .astype(np.int8)
    )

    return pd.DataFrame(hotspot_data, index=X.index)




In [7]:
class FeaturesEWithHotspot3AndPositionBins(FeaturesEWithPositionBins):
    def transform(self, X):
        base = make_features_E_sparse(X)
        hotspot = csr_matrix(make_hotspot_features(X).to_numpy(dtype=np.float32) * 3)
        position = self.position_transformer_.transform(X)
        return hstack([base, hotspot, position], format='csr', dtype=np.float32)

pipeline_hotspot3_pos_LR = Pipeline([
    ('features', FeaturesEWithHotspot3AndPositionBins(bin_size=50, min_patients=5)),
    ('model', LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000, solver='lbfgs'))
])


## 3. LightGBM: E + 정확한 `유전자_아미노산번호` (학습 Fold에서만 빈도 선택)

In [8]:
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from collections import Counter

# 변이 토큰의 시작에 있는 단일 영문자 + 숫자만 인식: V600E -> V600, R132H -> R132.
# 468_469LG처럼 숫자로 시작하거나 복잡한 표기는 의도적으로 제외.
EXACT_POSITION_RE = re.compile(r'^([A-Za-z])(\d+)')

def patient_exact_positions(X):
    result = []
    for row in X.itertuples(index=False, name=None):
        found = set()
        for gene, value in zip(X.columns, row):
            if gene in ('ID', 'SUBCLASS') or pd.isna(value) or str(value).strip().upper() == 'WT':
                continue
            for token in str(value).split():
                match = EXACT_POSITION_RE.match(token)
                if match:
                    found.add(f'{gene}_{match.group(1).upper()}{match.group(2)}')
        result.append(found)
    return result

class ExactPositionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_patients=5):
        self.min_patients = min_patients

    def fit(self, X, y=None):
        counts = Counter(key for keys in patient_exact_positions(X) for key in keys)
        self.selected_ = sorted(k for k,v in counts.items() if v >= self.min_patients)
        self.mapping_ = {key: i for i,key in enumerate(self.selected_)}
        return self

    def transform(self, X):
        if not hasattr(self, 'mapping_'):
            raise RuntimeError('ExactPositionTransformer.fit()을 먼저 실행하세요')
        rows, cols = [], []
        for i, keys in enumerate(patient_exact_positions(X)):
            for key in keys:
                j = self.mapping_.get(key)
                if j is not None:
                    rows.append(i); cols.append(j)
        return csr_matrix((np.ones(len(rows),dtype=np.float32),(rows,cols)),
                          shape=(len(X),len(self.selected_)),dtype=np.float32)

class FeaturesEWithExactPositions(BaseEstimator, TransformerMixin):
    def __init__(self, min_patients=5):
        self.min_patients = min_patients
    def fit(self, X, y=None):
        self.exact_ = ExactPositionTransformer(self.min_patients).fit(X)
        return self
    def transform(self, X):
        return hstack([make_features_E_sparse(X), self.exact_.transform(X)],
                      format='csr',dtype=np.float32)

lgbm_pipe = Pipeline([
    ('features', FeaturesEWithExactPositions(min_patients=5)),
    ('model', LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=15,
        min_child_samples=10, colsample_bytree=0.5, subsample=0.8,
        subsample_freq=1, class_weight='balanced', random_state=42,
        verbosity=-1, n_jobs=-1))
])
# 피처 선택이 fold 내부에서 이뤄지는지 확인하는 작은 예시
_demo = pd.DataFrame({'BRAF':['V600E','V600K','WT','V600E','V600E','V600E'], 'IDH1':['WT']*6})
assert ExactPositionTransformer(5).fit(_demo).selected_ == ['BRAF_V600']
print('Exact-position 변환기 테스트 통과')


Exact-position 변환기 테스트 통과


## 4. 개발 CV — 공통 Group 5-Fold, 기존 3번째 모델과 비교

In [9]:
K = len(le.classes_)
common = dict(n_estimators=250, max_depth=3, learning_rate=0.05,
              subsample=0.8, colsample_bytree=0.8,
              objective='multi:softprob', num_class=K, eval_metric='mlogloss',
              tree_method='hist', device=DEVICE, random_state=SEED, n_jobs=-1)
xgb = XGBClassifier(**common)
lr_E = LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000, solver='lbfgs')

def weight_array(y_part):
    counts = np.bincount(np.asarray(y_part,dtype=int),minlength=K)
    if not np.all(counts): raise ValueError('학습 Fold에 누락된 클래스가 있습니다')
    w = 1.0/np.sqrt(counts[np.asarray(y_part,dtype=int)])
    return (w/w.mean()).astype(np.float32)

def weighted_fit_params(y_part):
    return {'sample_weight':weight_array(y_part)}

def checked(p, rows):
    p = np.asarray(p,dtype=np.float64)
    assert p.shape==(rows,K) and np.isfinite(p).all()
    assert np.allclose(p.sum(axis=1),1,atol=1e-4)
    return p

def fold_scores(p, labels, cv_folds):
    return np.array([f1_score(labels[va],p[va].argmax(axis=1),average='macro',zero_division=0)
                     for _,va in cv_folds])

X_lr_dev = pd.DataFrame.sparse.from_spmatrix(
    csr_matrix(X_dev_E.to_numpy()),index=X_dev_E.index,columns=X_dev_E.columns)
cv_xgb = run_cv(xgb,X_dev_E,y_dev,folds=folds,fit_params_fn=weighted_fit_params)
cv_lr_E = run_cv(lr_E,X_lr_dev,y_dev,folds=folds)
cv_lr_pos = run_cv(pipeline_E_position_LR,X_dev_raw,y_dev,folds=folds)
cv_lr_hot = run_cv(pipeline_hotspot3_pos_LR,X_dev_raw,y_dev,folds=folds)
cv_lgbm = run_cv(lgbm_pipe,X_dev_raw,y_dev,folds=folds)

px = checked(cv_xgb['oof_proba'],len(y_dev))
pe = checked(cv_lr_E['oof_proba'],len(y_dev))
pp = checked(cv_lr_pos['oof_proba'],len(y_dev))
ph = checked(cv_lr_hot['oof_proba'],len(y_dev))
pl = checked(cv_lgbm['oof_proba'],len(y_dev))
old_oof = checked((px+pp)/2,len(y_dev))
new_oof = checked((px+pe+ph+pl)/4,len(y_dev))
old_scores=fold_scores(old_oof,y_dev,folds)
new_scores=fold_scores(new_oof,y_dev,folds)
cv_delta=float(new_scores.mean()-old_scores.mean())
results_dev=pd.DataFrame([
    {'model':'3번째: XGB + LR E/position (50:50)','mean':old_scores.mean(),'std':old_scores.std()},
    {'model':'4번째: XGB + LR E + Hotspot3/position LR + LGBM','mean':new_scores.mean(),'std':new_scores.std()},
])
display(results_dev)
print('개발 CV 개선:',f'{cv_delta:+.6f}', '| 5/5 개선:',bool(np.all(new_scores>old_scores)))
print('Fold별 변화:',np.round(new_scores-old_scores,6))


c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [20:00:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


fold 1: Macro F1 = 0.5049
fold 2: Macro F1 = 0.4753
fold 3: Macro F1 = 0.4806
fold 4: Macro F1 = 0.4810
fold 5: Macro F1 = 0.5048
CV Macro F1 = 0.4893 (std 0.0128)
fold 1: Macro F1 = 0.5025
fold 2: Macro F1 = 0.4644
fold 3: Macro F1 = 0.4740
fold 4: Macro F1 = 0.4757
fold 5: Macro F1 = 0.4889
CV Macro F1 = 0.4811 (std 0.0132)
fold 1: Macro F1 = 0.5133
fold 2: Macro F1 = 0.4652
fold 3: Macro F1 = 0.4629
fold 4: Macro F1 = 0.4909
fold 5: Macro F1 = 0.4982
CV Macro F1 = 0.4861 (std 0.0194)
fold 1: Macro F1 = 0.5033
fold 2: Macro F1 = 0.4761
fold 3: Macro F1 = 0.4614
fold 4: Macro F1 = 0.4881
fold 5: Macro F1 = 0.5059
CV Macro F1 = 0.4870 (std 0.0167)
fold 1: Macro F1 = 0.4893
fold 2: Macro F1 = 0.4659
fold 3: Macro F1 = 0.4726
fold 4: Macro F1 = 0.4883
fold 5: Macro F1 = 0.4761
CV Macro F1 = 0.4784 (std 0.0091)


,model,mean,std
0,3번째: XGB + LR E/position (50:50),0.506699,0.017581
1,4번째: XGB + LR E + Hotspot3/position LR + LGBM,0.500354,0.009803


개발 CV 개선: -0.006345 | 5/5 개선: False
Fold별 변화: [-0.007844  0.002077  0.001678 -0.001327 -0.026308]


## 5. 고정 홀드아웃 — 3번째 대비 **개발 CV와 홀드아웃 모두 개선** 확인

In [20]:
assert len(dev_idx)==4960 and len(hold_idx)==1241
raw_groups=pd.util.hash_pandas_object(X_raw.fillna('<MISSING>').astype(str),index=False).to_numpy()
assert not set(raw_groups[dev_idx]) & set(raw_groups[hold_idx]), '개발/홀드아웃 원본 중복 발견'
X_dev_E_hold, X_hold_E=X_E.iloc[dev_idx],X_E.iloc[hold_idx]
X_dev_raw_hold,X_hold_raw=X_raw.iloc[dev_idx],X_raw.iloc[hold_idx]
y_hold=np.asarray(y)[hold_idx]
mx=clone(xgb).fit(X_dev_E_hold,y_dev,sample_weight=weight_array(y_dev))
mp=clone(pipeline_E_position_LR).fit(X_dev_raw_hold,y_dev)
me=clone(lr_E).fit(csr_matrix(X_dev_E_hold.to_numpy()),y_dev)
mh=clone(pipeline_hotspot3_pos_LR).fit(X_dev_raw_hold,y_dev)
ml=clone(lgbm_pipe).fit(X_dev_raw_hold,y_dev)
for m in (mx,mp,me,mh,ml):
    assert np.array_equal(m.classes_,np.arange(K)), '클래스 순서 불일치'
p_x=checked(mx.predict_proba(X_hold_E),len(y_hold))
p_p=checked(mp.predict_proba(X_hold_raw),len(y_hold))
p_e=checked(me.predict_proba(csr_matrix(X_hold_E.to_numpy())),len(y_hold))
p_h=checked(mh.predict_proba(X_hold_raw),len(y_hold))
p_l=checked(ml.predict_proba(X_hold_raw),len(y_hold))
old_hold=float(f1_score(y_hold,((p_x+p_p)/2).argmax(axis=1),average='macro',zero_division=0))
new_hold=float(f1_score(y_hold,((p_x+p_e+p_h+p_l)/4).argmax(axis=1),average='macro',zero_division=0))
hold_delta=new_hold-old_hold
candidate_pass=(cv_delta>0) and (hold_delta>0)
display(pd.DataFrame([
    {'candidate':'3번째 50:50','dev_cv_mean':old_scores.mean(),'holdout_macro_f1':old_hold},
    {'candidate':'4번째 25:25:25:25','dev_cv_mean':new_scores.mean(),'holdout_macro_f1':new_hold}
]))
print(f'CV 변화 {cv_delta:+.6f} | 홀드아웃 변화 {hold_delta:+.6f}')
print('결과:', '두 지표 개선 — 최종 산출물 생성 가능' if candidate_pass else '보류 — 전체 OOF/Test 생성 금지')


,candidate,dev_cv_mean,holdout_macro_f1
0,3번째 50:50,0.506699,0.504429
1,4번째 25:25:25:25,0.500354,0.513594


CV 변화 -0.006345 | 홀드아웃 변화 +0.009165
결과: 보류 — 전체 OOF/Test 생성 금지


## 6. 전체 OOF/Test — 승인 후보만 실행

`RUN_FULL_CV=True` 설정 후 실행. 전체 Train 공통 Group 5-Fold로 OOF 생성. XGBoost Test는 08번처럼 전체 Train 재학습, LR 2개와 LightGBM Test는 5-Fold 예측 평균. **새 피처 선택은 각 학습 Fold 안에서만 수행**. Test는 추론 외 통계/분포 확인에 사용하지 않음.

예측 파일 재사용 시 클래스·형상뿐 아니라 **현재 실험의 메타데이터 설정을 확인**. 08번 XGB 기존 예측은 클래스 및 설정이 일치할 때 재사용 가능하지만, 다른 모델의 OOF를 대체하지 않음.


In [19]:
import os, tempfile
PRED=ROOT/'experiments'/'preds'
PRED.mkdir(parents=True,exist_ok=True)
N=len(y); NT=len(X_test_raw)
X_lr_full=pd.DataFrame.sparse.from_spmatrix(csr_matrix(X_E.to_numpy()),index=X_E.index,columns=X_E.columns)
XGB_STEM='A_model__xgb_E_depth3_invsqrt_position_ensemble_1110'
LR_E_STEM='A_model__lr_E_4ensemble_1110'
HOT_STEM='A_model__lr_E_hotspot3_position_4ensemble_1110'
LGB_STEM='A_model__lgbm_E_exact_position_4ensemble_1110'
ENS_STEM='A_model__ensemble_E_xgb_lr_hotspot3_lgbm_25252525_1110'

def valid_proba(a,rows,title):
    a=checked(a,rows)
    return a

def load_saved_xgb(kind,rows):
    path=PRED/f'{XGB_STEM}__{kind}.npy'
    meta=PRED/f'{XGB_STEM}__meta.json'
    if not path.exists(): return None
    if not meta.exists():
        print('기존 XGB 메타 없음 — 재계산:',path.name);return None
    old=json.loads(meta.read_text(encoding='utf-8'))
    if list(map(str,old.get('classes',[])))!=list(map(str,le.classes_)):
        raise ValueError('기존 XGB 클래스 순서 불일치')
    if 'weighted XGB E depth3' not in str(old.get('model','')):
        print('기존 XGB 모델 설정 확인 불가 — 재계산:',path.name);return None
    print('기존 XGB 파일 재사용:',path.name)
    return valid_proba(np.load(path,allow_pickle=False),rows,path.name)

if not RUN_FULL_CV:
    print('전체 OOF/Test 생성 대기: RUN_FULL_CV=True 설정')
else:
    if not candidate_pass:
        raise RuntimeError('개발 CV와 홀드아웃 모두 개선되지 않았습니다.')
    full_folds=list(get_folds(y,X=X_raw,classes=le.classes_))
    seen=np.zeros(N,dtype=int)
    for tr,va in full_folds: seen[va]+=1
    assert len(full_folds)==5 and np.all(seen==1)
    xgb_oof=load_saved_xgb('oof',N)
    xgb_test=load_saved_xgb('test',NT)
    if xgb_oof is None:
        xgb_oof=valid_proba(run_cv(xgb,X_E,y,folds=full_folds,
                        fit_params_fn=weighted_fit_params)['oof_proba'],N,'XGB OOF')
    if xgb_test is None:
        final_xgb=clone(xgb).fit(X_E,y,sample_weight=weight_array(y))
        assert np.array_equal(final_xgb.classes_,np.arange(K))
        xgb_test=valid_proba(final_xgb.predict_proba(X_test_E),NT,'XGB Test')

    # LR E / Hotspot LR / LGBM: fold별 학습·추론을 한 번만 수행
    model_defs={
        LR_E_STEM:(lr_E,X_lr_full, pd.DataFrame.sparse.from_spmatrix(
            csr_matrix(X_test_E.to_numpy()),columns=X_test_E.columns)),
        HOT_STEM:(pipeline_hotspot3_pos_LR,X_raw,X_test_raw),
        LGB_STEM:(lgbm_pipe,X_raw,X_test_raw)
    }
    full_outputs={}
    for stem,(model,X_in,X_test_in) in model_defs.items():
        oof=np.zeros((N,K),dtype=np.float64)
        tests=[]
        for i,(tr,va) in enumerate(full_folds,1):
            m=clone(model).fit(X_in.iloc[tr],np.asarray(y)[tr])
            assert np.array_equal(m.classes_,np.arange(K)),f'{stem} fold {i} 클래스 순서 불일치'
            oof[va]=valid_proba(m.predict_proba(X_in.iloc[va]),len(va),stem+' OOF')
            tests.append(valid_proba(m.predict_proba(X_test_in),NT,stem+' Test'))
            print(stem,'fold',i,'/5 완료')
        full_outputs[stem]=(valid_proba(oof,N,stem+' OOF'),
                            valid_proba(np.mean(tests,axis=0),NT,stem+' Test'))
    lr_oof,lr_test=full_outputs[LR_E_STEM]
    hot_oof,hot_test=full_outputs[HOT_STEM]
    lgb_oof,lgb_test=full_outputs[LGB_STEM]
    ens_oof=valid_proba((xgb_oof+lr_oof+hot_oof+lgb_oof)/4,N,'4 Ensemble OOF')
    ens_test=valid_proba((xgb_test+lr_test+hot_test+lgb_test)/4,NT,'4 Ensemble Test')
    print('전체 OOF Macro F1:',f1_score(y,ens_oof.argmax(axis=1),average='macro',zero_division=0))


RuntimeError: 개발 CV와 홀드아웃 모두 개선되지 않았습니다.

## 7. OOF/Test `.npy` + `.json` 저장 — `SAVE_ARTIFACTS=True` (CSV 생성 없음)

In [14]:
def save_array(stem,kind,arr):
    path=PRED/f'{stem}__{kind}.npy'
    arr=np.asarray(valid_proba(arr,len(arr),path.name),dtype=np.float32)
    if path.exists():
        existing=np.load(path,allow_pickle=False)
        if existing.shape!=arr.shape or not np.allclose(existing,arr,atol=1e-5):
            raise ValueError('기존 파일과 결과 불일치 — 덮어쓰기 중단: '+str(path))
        print('기존 동일 파일 유지:',path.name)
        return
    fd,tmp=tempfile.mkstemp(dir=PRED,suffix='.npy');os.close(fd)
    try:
        np.save(tmp,arr);os.replace(tmp,path)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)
    print('저장:',path.name)

def save_meta(stem,extra):
    path=PRED/f'{stem}__meta.json'
    obj={'classes':list(map(str,le.classes_)),'train_rows':N,'test_rows':NT,
         'folds':5,'seed':int(SEED),**extra}
    if path.exists():
        existing=json.loads(path.read_text(encoding='utf-8'))
        if existing!=obj:
            # 이전 08번 XGB 메타는 스키마가 다르므로 변경하지 않고 그대로 유지
            if stem==XGB_STEM and list(map(str,existing.get('classes',[])))==obj['classes']:
                print('기존 08 XGB 메타 유지:',path.name);return
            raise ValueError('기존 메타와 실험 설정 불일치: '+str(path))
        print('기존 동일 메타 유지:',path.name);return
    path.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')
    print('메타 저장:',path.name)

if not SAVE_ARTIFACTS:
    print('저장 대기: SAVE_ARTIFACTS=True 설정')
elif not RUN_FULL_CV or not candidate_pass:
    raise RuntimeError('RUN_FULL_CV=True 및 두 지표 개선 필요')
else:
    items=[
      (XGB_STEM,xgb_oof,xgb_test,{'model':'weighted XGB E depth3 250 trees','sample_weight':'inverse sqrt frequency'}),
      (LR_E_STEM,lr_oof,lr_test,{'model':'LR E','C':0.1}),
      (HOT_STEM,hot_oof,hot_test,{'model':'LR E + hotspot x3 + position bins','bin_size':50,'min_patients':5,'hotspots':36}),
      (LGB_STEM,lgb_oof,lgb_test,{'model':'LightGBM E + exact gene AA position','min_patients':5,
         'params':lgbm_pipe.named_steps['model'].get_params()}),
      (ENS_STEM,ens_oof,ens_test,{'model':'4-model equal probability ensemble',
         'weights':[0.25]*4,'sources':[XGB_STEM,LR_E_STEM,HOT_STEM,LGB_STEM]})
    ]
    for stem,oof,tst,extra in items:
        save_array(stem,'oof',oof)
        save_array(stem,'test',tst)
        save_meta(stem,extra)
    print('OOF/Test 및 메타 저장 완료 — 제출 CSV는 생성하지 않았습니다.')


저장 대기: SAVE_ARTIFACTS=True 설정


## 8. PR 체크

- `git rm --cached submissions/A_model__ensemble_E_xgb_invsqrt_position_lr_5050_1110__submission.csv` 변경을 이번 PR에 포함.
- 09번 노트북, 새 OOF/Test `.npy`, `.json`만 Commit. **`submissions/` 파일을 `git add -f` 하지 않음**.
- `git diff --cached --check` 및 노트북 Run All 검증.
- `RUN_FULL_CV=False`, `SAVE_ARTIFACTS=False`가 기본값. 두 지표 개선 시에만 True로 전환.
- 개발 CV와 홀드아웃, 전체 OOF 점수는 평가 데이터가 달라 직접 동등 비교하지 않음.
- Test는 최종 추론 전용. 클래스 순서/확률 합 검증, Fold 학습부분에서만 위치 피처 선택.
